# Expansion Planning & Sector Coupling

:::{note}
**On the workshop JupyterHub, everything is already installed** - just log in and run the cells below, nothing to set up.

No hub access? You can also run this notebook in your browser on [Google Colab](https://colab.research.google.com/github/pypsalabs/workshop-202610/blob/main/berlin/03-pypsa-sectors-expansion.ipynb) (a Google account is required). On Colab, first run this in a cell at the top:

```sh
!pip install pypsa
```
:::

## Problem description

To explore the capacity expansion problem with sector-coupling options, let's model a "greenfield" energy system with:
- electricity demand (here: Germany 2015)
- several technologies for electricity generation (wind, solar, and gas for peak load)
- hydrogen consumption (e.g., an offtaker in the industrial sector)
- hydrogen production from electrolysis
- hydrogen storage
- hydrogen fuel cell
- heat demand
- heat pumps and resistive heaters

In [ ]:
import pandas as pd
import pypsa

pypsa.options.params.optimize.include_objective_constant = False

pd.options.plotting.backend = "plotly"

## Prepare technology data

The PyPSA project maintains a database (https://github.com/PyPSA/technology-data) which collects assumptions and projections for energy system technologies (such as costs, efficiencies, lifetimes, etc.) for given years, which we can load into a `pandas.DataFrame`.

Reading this data into a usable `pandas.DataFrame` requires some pre-processing (e.g. converting units, setting defaults, re-arranging dimensions):

In [ ]:
YEAR = 2030
# technology-data pinned to release v0.13.4 (October 2025)
# so that the numbers in this notebook are reproducible across builds
VERSION = "v0.13.4"
url = f"https://raw.githubusercontent.com/PyPSA/technology-data/{VERSION}/outputs/costs_{YEAR}.csv"
costs = pd.read_csv(url, index_col=[0, 1])

In [ ]:
costs.loc[costs.unit.str.contains("/kW"), "value"] *= 1e3
costs.unit = costs.unit.str.replace("/kW", "/MW")

defaults = {
    "FOM": 0,
    "VOM": 0,
    "efficiency": 1,
    "fuel": 0,
    "investment": 0,
    "lifetime": 25,
    "CO2 intensity": 0,
    "discount rate": 0.07,
}
costs = costs.value.unstack().fillna(defaults)

costs.at["OCGT", "fuel"] = costs.at["gas", "fuel"]
costs.at["OCGT", "CO2 intensity"] = costs.at["gas", "CO2 intensity"]

In [ ]:
costs.head()

Let's also write a small utility _function_ that calculates the **annuity** to annualise investment costs. The formula is

$$
a(r, n) = \frac{r}{1-(1+r)^{-n}}
$$
where $r$ is the discount rate and $n$ is the lifetime. If $r=0$, the annuity simplifies to $a(0,n) = \frac{1}{n}$. If $n=\infty$, the annuity simplifies to $a(r, \infty) = r$.

:::{note}
PyPSA ships the same function as `pypsa.costs.annuity` — here we write it ourselves to see what happens.
:::

In [ ]:
def annuity_factor(r, n):
    return 1 / n if r == 0 else r / (1 - (1 + r) ** -n)

The resulting `annuity_factor` (often called capital recovery factor) represents the factor that, when multiplied by the initial capital cost, yields the equivalent annual cost accounting for the time value of money:

In [ ]:
annuity_factor(0.07, 20)

Based on this, we can calculate the short-term marginal generation costs (€/MWh), named `marginal_cost` in PyPSA:

In [ ]:
costs["marginal_cost"] = costs["VOM"] + costs["fuel"] / costs["efficiency"]

and the annualised investment costs (`capital_cost` in PyPSA terms, €/MW/a). The FOM cost is expressed as a percentage of the overnight investment cost per year, and thus can be added to the annuity factor when calculating the annualised capital cost:

In [ ]:
annuity = costs.apply(lambda x: annuity_factor(0.07, x["lifetime"]), axis=1)

In [ ]:
costs["capital_cost"] = (annuity + costs["FOM"] / 100) * costs["investment"]

## Prepare load and renewable generation time-series

We are also going to need some time series for wind, solar and load. For this example, we use time series for Germany in the year 2015 — the same ones we used in the `pandas` tutorial in the annex.

In [ ]:
# Backup: https://cloud.pypsalabs.org/s/FSJ3qwQjqCxcWXQ/download/time-series-sample.csv
url = "https://raw.githubusercontent.com/pypsalabs/workshop-202610/main/berlin/data/time-series-sample.csv"

ts = pd.read_csv(url, index_col=0, parse_dates=True)

In [ ]:
ts.head()

Let's also convert the load time series from GW to MW, the base unit of PyPSA:

In [ ]:
ts.load *= 1e3

Optionally, we can downscale temporal resolution of the time series to save some computation time:

In [ ]:
# here we sample only every fourth hour:
resolution = 4
ts = ts.resample(f"{resolution}h").first()

## Build our energy system to simulate

As always, let's initialize an empty network:

In [ ]:
n = pypsa.Network()

Then, we add a single electricity bus...

In [ ]:
n.add("Bus", "electricity")

...and tell the `pypsa.Network` object `n` what the [snapshots](https://docs.pypsa.org/latest/user-guide/design/#snapshots) of the model will be using the utility function `n.set_snapshots()`.

In [ ]:
n.set_snapshots(ts.index)

In [ ]:
n.snapshots

If we resampled the time series above, we need to adjust the weighting of the snapshots (i.e. how many hours they represent). We can do that with `n.snapshot_weightings`:

In [ ]:
n.snapshot_weightings.head(3)

In [ ]:
n.snapshot_weightings.loc[:, :] = resolution

In [ ]:
n.snapshot_weightings.head(3)

### Adding components for electricity part

Then, we add all the technologies we are going to include as carriers.

In [ ]:
carriers = [
    "onwind",
    "offwind",
    "solar",
    "OCGT",
    "hydrogen storage underground",
]

n.add(
    "Carrier",
    carriers,
    color=["dodgerblue", "aquamarine", "gold", "indianred", "magenta"],
    co2_emissions=[costs.at[c, "CO2 intensity"] for c in carriers],
)

Next, we add the demand time series to the model.

In [ ]:
n.add(
    "Load",
    "demand",
    bus="electricity",
    p_set=ts.load,
)

Let's have a check whether the data was read-in correctly.

In [ ]:
n.loads_t.p_set.plot(labels={"value": "Load [MW]"})

We are going to add one dispatchable generation technology to the model. This is an open-cycle gas turbine (OCGT) with CO$_2$ emissions of 0.2 t/MWh$_{th}$.

In [ ]:
n.add(
    "Generator",
    "OCGT",
    bus="electricity",
    carrier="OCGT",
    capital_cost=costs.at["OCGT", "capital_cost"],
    marginal_cost=costs.at["OCGT", "marginal_cost"],
    efficiency=costs.at["OCGT", "efficiency"],
    p_nom_extendable=True,
)

Adding the variable renewable generators works almost identically, but we also need to supply the capacity factors to the model via the attribute `p_max_pu`.

In [ ]:
for tech in ["onwind", "offwind", "solar"]:
    n.add(
        "Generator",
        tech,
        bus="electricity",
        carrier=tech,
        p_max_pu=ts[tech],
        capital_cost=costs.at[tech, "capital_cost"],
        marginal_cost=costs.at[tech, "marginal_cost"],
        efficiency=costs.at[tech, "efficiency"],
        p_nom_extendable=True,
    )

In [ ]:
# Making sure the capacity factors are read-in correctly
n.generators_t.p_max_pu.loc["2015-03"].plot(labels={"value": "Capacity Factor [p.u.]"})

### Adding components for hydrogen part

Add a dedicated `Bus` for the hydrogen energy carrier:

In [ ]:
n.add("Bus", "hydrogen", carrier="hydrogen")

Add a `Link` for the hydrogen electrolysis:

In [ ]:
n.add(
    "Link",
    "electrolysis",
    bus0="electricity",
    bus1="hydrogen",
    carrier="electrolysis",
    p_nom_extendable=True,
    efficiency=0.7,
    capital_cost=50e3,  # €/MW/a
)

:::{note}
Some of the sector-coupling technologies might have multiple outputs (e.g. CHP plants producing heat and power). PyPSA can automatically handle links that have more than one input (`bus0`)
and/or output (i.e. `bus1`, `bus2`, `bus3`) with a given efficiency (`efficiency`, `efficiency2`, `efficiency3`).
:::

Add a `Link` for the fuel cell which reconverts hydrogen to electricity:

In [ ]:
n.add(
    "Link",
    "fuel cell",
    bus0="hydrogen",
    bus1="electricity",
    carrier="fuel cell",
    p_nom_extendable=True,
    efficiency=0.5,
    capital_cost=120e3,  # €/MW/a
)

Add a `Store` for the hydrogen storage:

In [ ]:
n.add(
    "Store",
    "hydrogen storage",
    bus="hydrogen",
    carrier="hydrogen storage",
    capital_cost=140,  # €/MWh/a
    e_nom_extendable=True,
    e_cyclic=True,  # cyclic state of charge
)

To model an industrial hydrogen offtaker, we add also a hydrogen demand to the hydrogen bus.

In the example below, we add a hydrogen demand such that it equals ~25% of the electricity demand (in MWh$_{H_2}$):

In [ ]:
n.add(
    "Load", "hydrogen demand", bus="hydrogen", carrier="hydrogen", p_set=19500
)  # MWh_H2/h

### We are now ready to solve the model

In [ ]:
n.optimize(solver_name="highs", log_to_console=False)
# ~66 seconds for 1H temporal resolution (8760 snapshots)
# ~7 seconds for 4H temporal resolution (2190 snapshots)

### Exploring model results

The total system cost in billion Euros per year:

In [ ]:
n.objective / 1e9

`n.statistics()` provides an informative overview of the model results.

See documentation: https://docs.pypsa.org/latest/user-guide/statistics/

In [ ]:
n.statistics()

We can use `n.statistics()` to get a quick overview of optimised capacities across all components:

In [ ]:
n.statistics.expanded_capacity().div(1e3).round(1)  # GW

You can also use `n.statistics()` to promptly get an energy balance for the complete system or even any specific bus:

In [ ]:
n.statistics.energy_balance(aggregate_time=False)

In [ ]:
n.statistics.energy_balance(aggregate_time=False, bus_carrier="hydrogen").div(
    1e3
).groupby("carrier").sum().T.plot()

Possibly, we are also interested in the total emissions:

In [ ]:
emissions = (
    n.generators_t.p
    / n.generators.efficiency
    * n.generators.carrier.map(n.carriers.co2_emissions)
)  # t/h

In [ ]:
n.snapshot_weightings.generators @ emissions.sum(axis=1).div(1e6)  # Mt

## Adding emission limits

The gas power plant offers sufficient and cheap enough backup capacity to run in periods of low wind and solar generation. But what happens if this source of flexibility disappears? Let's model a 100% renewable electricity system by adding a CO$_2$ emission limit as global constraint:

In [ ]:
n.add(
    "GlobalConstraint",
    "CO2Limit",
    carrier_attribute="co2_emissions",
    sense="<=",
    constant=0,
)

When we run the model now...

In [ ]:
n.optimize(solver_name="highs", log_to_console=False)

The total system cost (billion Euros per year) is now higher than before:

In [ ]:
n.objective / 1e9

The optimal capacity mix now does not include any gas power plants and includes onshore wind and fuel cell technologies:

In [ ]:
n.statistics.expanded_capacity().div(1e3).round(1)  # GW

Fuel cell technology steps in hours with low wind and solar generation:

In [ ]:
n.statistics.energy_balance(aggregate_time=False, bus_carrier="AC").div(1e3).groupby(
    "carrier"
).sum().T.plot()

In [ ]:
n.statistics.energy_balance(aggregate_time=False, bus_carrier="hydrogen").div(
    1e3
).groupby("carrier").sum().T.plot()

In [ ]:
n.stores_t.e.plot()

Total emissions are now zero:

In [ ]:
emissions = (
    n.generators_t.p
    / n.generators.efficiency
    * n.generators.carrier.map(n.carriers.co2_emissions)
)  # t/h

In [ ]:
n.snapshot_weightings.generators @ emissions.sum(axis=1).div(1e6)  # Mt

## Adding the heating sector

So far, our model covers electricity and hydrogen. Let's now couple the heating sector: we add a heat demand and two technologies that convert electricity to heat, heat pumps and resistive heaters. The emissions limit from above stays in place, so all heat must come from renewable electricity.

For modelling simple heating systems, we create another bus and connect a load with the heat demand time series to it.

In [ ]:
n.add("Bus", "heat", carrier="heat")

The heat demand time series (in MW) comes at 4-hourly resolution, matching our snapshots:

In [ ]:
# Backup: https://tubcloud.tu-berlin.de/s/mSkHERH8fJCKNXx/download/heat-load-example.csv
url = "https://raw.githubusercontent.com/pypsalabs/workshop-202610/main/berlin/data/heat-load-example.csv"

heat_load = pd.read_csv(url, index_col=0, parse_dates=True).squeeze()

In [ ]:
heat_load.head()

In [ ]:
n.add("Load", "heat demand", bus="heat", carrier="heat", p_set=heat_load)

Let's compare the electricity and heat demand:

In [ ]:
n.loads_t.p_set.plot(labels={"value": "Load [MW]"})

What is missing now are the components to supply heat to the `"heat"` bus, such as heat pumps, resistive heaters or combined heat and power plants (CHPs). Here, we focus on heat pumps and resistive heaters.

### Heat pumps

To model heat pumps, we first have to calculate the **coefficient of performance (COP)** profile based on the temperature profile of the heat source. The COP is the amount of heat a heat pump delivers per unit of electricity.

In the example below, we calculate the COP for an air-sourced heat pump with a sink temperature of 55 °C and a population-weighted ambient temperature profile for Germany.

The heat pump performance is given by the following function:

$$
COP(\Delta T) = 6.81 - 0.121 \Delta T + 0.00063 \Delta T^2
$$
where $\Delta T = T_{sink} - T_{source}$.

In [ ]:
def cop(t_source, t_sink=55):
    delta_t = t_sink - t_source
    return 6.81 - 0.121 * delta_t + 0.000630 * delta_t**2

We take the ambient temperature for 2015 from [Renewables.ninja](https://www.renewables.ninja) and downsample it like the other time series:

In [ ]:
# Backup: https://tubcloud.tu-berlin.de/s/S4jRAQMP5Te96jW/download/ninja_weather_country_DE_merra-2_population_weighted.csv
url = "https://raw.githubusercontent.com/pypsalabs/workshop-202610/main/berlin/data/ninja_weather_country_DE_merra-2_population_weighted.csv"

weather = pd.read_csv(url, skiprows=2, index_col=0, parse_dates=True)
temp = weather.loc["2015", "temperature"].resample(f"{resolution}h").first()

The COP drops as the ambient temperature falls:

In [ ]:
pd.DataFrame({"temperature": temp, "COP": cop(temp)}).plot.scatter(
    x="temperature", y="COP", labels={"temperature": "Temperature [°C]"}
)

So the COP is lowest in winter, exactly when heat demand peaks:

In [ ]:
cop(temp).plot(labels={"value": "COP [-]"})

Once we have calculated the heat pump coefficient of performance, we can add the heat pump to the network as a `Link`. We use the parameter `efficiency` to incorporate the time-varying COP. Note that the capacity of a `Link` refers to its input (`bus0`), here MW of electricity. The technology data gives costs per MW of heat output, so we multiply them by the nominal COP to get costs per MW of electricity input.

In [ ]:
n.add(
    "Link",
    "heat pump",
    bus0="electricity",
    bus1="heat",
    carrier="heat pump",
    p_nom_extendable=True,
    efficiency=cop(temp),
    capital_cost=costs.at["decentral air-sourced heat pump", "capital_cost"]
    * costs.at["decentral air-sourced heat pump", "efficiency"],  # €/MW_el/a
)

### Resistive heaters

Let's also add a resistive heater as backup technology. Its efficiency is constant and well below the COP of a heat pump, but it is much cheaper to build:

In [ ]:
n.add(
    "Link",
    "resistive heater",
    bus0="electricity",
    bus1="heat",
    carrier="resistive heater",
    p_nom_extendable=True,
    efficiency=costs.at["decentral resistive heater", "efficiency"],
    capital_cost=costs.at["decentral resistive heater", "capital_cost"]
    * costs.at["decentral resistive heater", "efficiency"],  # €/MW_el/a
)

### Solving the model with heating

In [ ]:
n.optimize(solver_name="highs", log_to_console=False)

The total system cost (billion Euros per year) increases, since we now also supply heat:

In [ ]:
n.objective / 1e9

The optimal capacity mix now includes heat pumps and resistive heaters (link capacities in GW of electricity input). Fuel cell capacity more than doubles, since heating adds electricity demand in winter, when solar output is low:

In [ ]:
n.statistics.expanded_capacity().div(1e3).round(1)  # GW

The annual heat balance (positive is supply, negative is consumption) shows that heat pumps supply nearly all heat:

In [ ]:
n.statistics.energy_balance(bus_carrier="heat").div(1e6).round(1)  # TWh

Heat pumps are expensive to build, so the model does not size them for the coldest hours. Cheap resistive heaters cover these winter peaks, while heat pumps run at full capacity:

In [ ]:
n.statistics.energy_balance(aggregate_time=False, bus_carrier="heat").div(1e3).groupby(
    "carrier"
).sum().T.plot()

On the electricity bus, heat pumps now appear as a large additional consumer, mostly in winter. Fuel cells, fed by stored hydrogen, run more in winter to cover this extra demand:

In [ ]:
n.statistics.energy_balance(aggregate_time=False, bus_carrier="AC").div(1e3).groupby(
    "carrier"
).sum().T.plot()

## Exercises

Explore the model by changing the assumptions and available technologies. Here are a few inspirations, which you do not have to follow in order:

**Task 1: Optimistic Costs** Rerun the model with cost assumptions for 2050. You can change the year when loading the technology data.

**Task 2: Storage Constraints** What if hydrogen storage cannot be expanded? You can remove components with `n.remove("Store", "hydrogen storage")`. How does the system compensate?

**Task 3: Renewable Constraints** What if you can either only build solar or only build wind? You can remove components with `n.remove("Generator", "ComponentName")`.

**Task 4: Nuclear** Add nuclear as another dispatchable low-emission generator (modelled similarly to the OCGT generator). Perform a sensitivity analysis trying to answer how low the capital cost of a nuclear plant would need to be to be chosen in the cost-optimal mix.

**Task 5: Fossil Fuel Crisis** Observe how the total system cost and composition of technologies changes with increasing gas prices (which could be a result of an energy crisis or carbon pricing). You can change the gas price by adjusting the `marginal_cost` of the OCGT generator.